# Docling sur CPU : conversion de PDF (CV) en Markdown / JSON

Ce notebook teste **Docling** (IBM) **sur CPU uniquement**, avec quatre modes :

| Mode | Pour quels PDF | Modèles utilisés | Vitesse CPU |
|---|---|---|---|
| **A. Standard sans OCR** | PDF natifs (texte sélectionnable) | mise en page + tableaux | rapide |
| **B. Standard avec OCR** | PDF scannés / images | mise en page + tableaux + EasyOCR | moyen |
| **C. VLM granite-docling-258M** | tous | 1 petit modèle vision-langage | lent |
| **D. Extraction JSON (bêta)** | tous | NuExtract-2.0-2B intégré à Docling | très lent |

Pour chaque mode : **temps d'exécution**, **temps par étape**, **pic de RAM**, **choix des pages** et sauvegarde des résultats.

> Exécutez les cellules 1 à 4 une fois, puis les modes que vous voulez tester. Les modèles se téléchargent automatiquement au premier usage (Hugging Face).

## 1. Installation

In [ ]:
!pip install -q docling psutil pymupdf

## 2. Forcer le CPU + imports + outils de mesure

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""      # masque le GPU : tout tourne sur CPU

import json, time, threading, gc
import psutil, pymupdf
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (PdfPipelineOptions, TableFormerMode,
                                                EasyOcrOptions, VlmPipelineOptions)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.settings import settings
try:
    from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice
except ImportError:  # anciennes versions de Docling
    from docling.datamodel.pipeline_options import AcceleratorOptions, AcceleratorDevice

settings.debug.profile_pipeline_timings = True   # active les temps par étape

N_THREADS = os.cpu_count()
ACCEL = AcceleratorOptions(num_threads=N_THREADS, device=AcceleratorDevice.CPU)
PROC = psutil.Process(os.getpid())

def ram_mo():
    return PROC.memory_info().rss / 1024**2

class SuiviRAM:
    """Mesure durée et pic de RAM d'un bloc `with` (échantillonnage toutes les 0,1 s)."""
    def __enter__(self):
        self.avant = self.pic = ram_mo()
        self._stop = threading.Event()
        self._th = threading.Thread(target=self._boucle, daemon=True); self._th.start()
        self.t0 = time.perf_counter(); return self
    def _boucle(self):
        while not self._stop.is_set():
            self.pic = max(self.pic, ram_mo()); time.sleep(0.1)
    def __exit__(self, *exc):
        self.duree = time.perf_counter() - self.t0
        self._stop.set(); self._th.join()
        self.apres = ram_mo(); self.pic = max(self.pic, self.apres)

RAM_DEPART = ram_mo()
print(f"CPU : {N_THREADS} coeurs | RAM machine : {psutil.virtual_memory().total/1024**3:.1f} Go "
      f"| RAM notebook : {RAM_DEPART:.0f} Mo")

## 3. Paramètres : fichier et pages

Docling traite une **plage continue** de pages : `"1"`, `"1-2"`, `"2-4"` ou `"all"`.

In [ ]:
PDF_PATH = "/content/cv2.pdf"
PAGES    = "1-2"                 # "1", "1-2", "all"
DOSSIER_SORTIE = "/content/docling_sorties"
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

def plage_pages(spec, pdf_path):
    with pymupdf.open(pdf_path) as d:
        n = len(d)
        natif = [bool(p.get_text().strip()) for p in d]
    if str(spec).strip().lower() == "all":
        debut, fin = 1, n
    elif "-" in str(spec):
        debut, fin = map(int, str(spec).split("-"))
    else:
        debut = fin = int(spec)
    if not (1 <= debut <= fin <= n):
        raise ValueError(f"Plage {spec} invalide : le PDF a {n} pages")
    return (debut, fin), n, natif

PAGE_RANGE, N_PAGES, NATIF = plage_pages(PAGES, PDF_PATH)
print(f"PDF : {N_PAGES} pages | pages traitées : {PAGE_RANGE[0]} à {PAGE_RANGE[1]}")
for i, ok in enumerate(NATIF, 1):
    print(f"  Page {i} : {'texte natif ✅' if ok else 'pas de texte ➜ scan, utilisez le mode B, C ou D'}")

## 4. Fonctions communes (conversion, mesures, sauvegarde)

In [ ]:
RESULTATS = {}

def temps_par_etape(conv_res):
    """Temps cumulés par étape du pipeline Docling (si le profilage est actif)."""
    etapes = {}
    for nom, item in (conv_res.timings or {}).items():
        try:
            etapes[nom] = round(sum(item.times), 2)
        except Exception:
            pass
    return etapes

def executer(nom_mode, converter_factory):
    """Crée le convertisseur (chargement des modèles) puis convertit, en mesurant tout."""
    gc.collect()
    with SuiviRAM() as m_init:
        converter = converter_factory()
        converter.initialize_pipeline(InputFormat.PDF)   # charge les modèles maintenant
    with SuiviRAM() as m_conv:
        res = converter.convert(PDF_PATH, page_range=PAGE_RANGE)
    doc = res.document
    n = PAGE_RANGE[1] - PAGE_RANGE[0] + 1

    markdown = doc.export_to_markdown()
    base = os.path.join(DOSSIER_SORTIE, nom_mode)
    with open(base + ".md", "w", encoding="utf-8") as f:
        f.write(markdown)
    with open(base + ".json", "w", encoding="utf-8") as f:
        json.dump(doc.export_to_dict(), f, ensure_ascii=False)

    stats = {
        "statut": str(res.status),
        "pages": n,
        "chargement_modeles_s": round(m_init.duree, 2),
        "conversion_s": round(m_conv.duree, 2),
        "par_page_s": round(m_conv.duree / n, 2),
        "total_s": round(m_init.duree + m_conv.duree, 2),
        "ram_apres_chargement_mo": round(m_init.apres),
        "ram_pic_mo": round(max(m_init.pic, m_conv.pic)),
        "ram_ajoutee_mo": round(max(m_init.pic, m_conv.pic) - RAM_DEPART),
        "caracteres_sortie": len(markdown),
        "etapes_s": temps_par_etape(res),
    }
    RESULTATS[nom_mode] = stats

    print(markdown[:3000] + ("\n..." if len(markdown) > 3000 else ""))
    print("\n----- Mesures -----")
    print(f"Statut : {stats['statut']}")
    print(f"⏱ Chargement des modèles : {stats['chargement_modeles_s']} s")
    print(f"⏱ Conversion             : {stats['conversion_s']} s ({stats['par_page_s']} s/page)")
    print(f"⏱ Total                  : {stats['total_s']} s")
    print(f"💾 RAM pic               : {stats['ram_pic_mo']} Mo (+{stats['ram_ajoutee_mo']} Mo par rapport au départ)")
    if stats["etapes_s"]:
        print("⏱ Par étape :", ", ".join(f"{k}={v}s" for k, v in stats["etapes_s"].items()))
    print(f"📁 Sorties : {base}.md et {base}.json")

    del converter, res
    gc.collect()
    return doc, stats

print("Prêt ✅")

## Mode A : pipeline standard **sans OCR** (PDF natifs, le plus rapide)

Docling lit le texte directement dans le PDF, détecte la mise en page (titres, listes, ordre de lecture) et la structure des tableaux.

In [ ]:
def converter_standard():
    o = PdfPipelineOptions()
    o.accelerator_options = ACCEL
    o.do_ocr = False
    o.do_table_structure = True
    o.table_structure_options.mode = TableFormerMode.FAST     # ACCURATE = plus précis, plus lent
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=o)})

doc_A, stats_A = executer("A_standard_sans_ocr", converter_standard)

## Mode B : pipeline standard **avec OCR** (PDF scannés)

EasyOCR lit le texte des images. Réglez `LANGUES_OCR` selon vos CV (EasyOCR : `"ar"` ne se combine qu'avec `"en"`).

In [ ]:
LANGUES_OCR = ["fr", "en"]
OCR_PAGE_ENTIERE = True      # True = OCR de toute la page (pour les scans)

def converter_ocr():
    o = PdfPipelineOptions()
    o.accelerator_options = ACCEL
    o.do_ocr = True
    ocr = EasyOcrOptions(lang=LANGUES_OCR, use_gpu=False)
    if OCR_PAGE_ENTIERE:
        try:
            from docling.datamodel.pipeline_options import OcrMode
            ocr.mode = OcrMode.FULL_PAGE
        except ImportError:                     # anciennes versions
            ocr.force_full_page_ocr = True
    o.ocr_options = ocr
    o.do_table_structure = True
    o.table_structure_options.mode = TableFormerMode.FAST
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=o)})

doc_B, stats_B = executer("B_standard_avec_ocr", converter_ocr)

## Mode C : VLM **granite-docling-258M** (un seul petit modèle vision-langage)

Le modèle « regarde » chaque page et produit directement la structure du document. Il gère bien les mises en page complexes, mais il est **lent sur CPU** : commencez avec **une seule page** (`PAGES = "1"`).

In [ ]:
from docling.pipeline.vlm_pipeline import VlmPipeline
from docling.datamodel import vlm_model_specs

def converter_vlm():
    o = VlmPipelineOptions(vlm_options=vlm_model_specs.GRANITEDOCLING_TRANSFORMERS)
    o.accelerator_options = ACCEL
    return DocumentConverter(format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_cls=VlmPipeline, pipeline_options=o)})

doc_C, stats_C = executer("C_vlm_granite_docling", converter_vlm)

## Mode D (bêta) : extraction **JSON selon un schéma** directement avec Docling

Docling intègre une API d'extraction avec **NuExtract-2.0-2B**. Elle renvoie un JSON **par page**, selon votre gabarit.
⚠️ Le modèle de 2 milliards de paramètres tourne dans `transformers` sans quantification : c'est **très lent sur CPU** et il faut plusieurs Go de RAM. Testez sur une seule page.

In [ ]:
from docling.document_extractor import DocumentExtractor

schema_json_experience = {
    "experience": [{
        "poste": "verbatim-string",
        "entreprise": "verbatim-string",
        "debut": "date-time",
        "fin": "date-time",
        "description": "string"
    }]
}

gc.collect()
with SuiviRAM() as m_init:
    extractor = DocumentExtractor(allowed_formats=[InputFormat.IMAGE, InputFormat.PDF])
with SuiviRAM() as m_ext:
    res_D = extractor.extract(source=PDF_PATH, template=schema_json_experience, page_range=PAGE_RANGE)

n = PAGE_RANGE[1] - PAGE_RANGE[0] + 1
for p in res_D.pages:
    print(f"--- Page {p.page_no} ---")
    print(json.dumps(p.extracted_data, indent=2, ensure_ascii=False))
    if p.errors:
        print("Erreurs :", p.errors)

RESULTATS["D_extraction_nuextract2b"] = {
    "statut": str(res_D.status), "pages": n,
    "chargement_modeles_s": round(m_init.duree, 2),
    "conversion_s": round(m_ext.duree, 2),
    "par_page_s": round(m_ext.duree / n, 2),
    "total_s": round(m_init.duree + m_ext.duree, 2),
    "ram_pic_mo": round(max(m_init.pic, m_ext.pic)),
    "ram_ajoutee_mo": round(max(m_init.pic, m_ext.pic) - RAM_DEPART),
}
with open(os.path.join(DOSSIER_SORTIE, "D_extraction.json"), "w", encoding="utf-8") as f:
    json.dump([{"page": p.page_no, "data": p.extracted_data} for p in res_D.pages], f,
              indent=2, ensure_ascii=False)
print(f"\n⏱ Extraction : {m_ext.duree:.1f} s ({m_ext.duree/n:.1f} s/page, modèle chargé pendant l'extraction) "
      f"| 💾 RAM pic : {max(m_init.pic, m_ext.pic):.0f} Mo")
del extractor; gc.collect()

## Récapitulatif des modes testés

In [ ]:
print(f"{'Mode':<28}{'Pages':>6}{'Chargement':>12}{'Conversion':>12}{'s/page':>9}{'RAM pic':>10}")
for nom, s in RESULTATS.items():
    print(f"{nom:<28}{s['pages']:>6}{s['chargement_modeles_s']:>11}s{s['conversion_s']:>11}s"
          f"{s['par_page_s']:>9}{s['ram_pic_mo']:>8}Mo")

with open(os.path.join(DOSSIER_SORTIE, "mesures.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTATS, f, indent=2, ensure_ascii=False)
print(f"\nMesures sauvegardées dans {DOSSIER_SORTIE}/mesures.json")

## Suite possible : Docling ➜ NuExtract3

Le Markdown produit par Docling (fichiers `.md`) conserve les titres et l'ordre de lecture : c'est une bonne entrée pour le notebook **NuExtract3 CPU**.
Il suffit d'y remplacer le texte PyMuPDF par :

```python
cv_text = open("/content/docling_sorties/A_standard_sans_ocr.md", encoding="utf-8").read()
```